# COTPT: Chain of Thought Per Token on Google Colab

This notebook demonstrates **per-token hidden deliberation with aggressive KV-cache eviction** using `cotpt`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedbarakat207/cotpt/blob/main/colab_demo.ipynb)

### What this notebook covers:
1. **Environment Setup**: Install dependencies on a Colab GPU runtime.
2. **Zero-Shot Inference**: Watch hidden deliberation generate thoughts per token and evict them from the KV cache in real-time.
3. **Fine-Tuning with RL**: Train with REINFORCE + Value Baseline + LoRA on reasoning data.
4. **Benchmarking**: Compare Accuracy (%), Peak KV-cache size, latency, and throughput across normal and COTPT conditions.
5. **Interactive Chat**: Chat with the model directly in the notebook.

## 1. Setup Environment
Make sure your Colab runtime is set to **GPU** (Runtime -> Change runtime type -> T4 GPU or better).

In [ ]:
!git clone https://github.com/ahmedbarakat207/cotpt.git
%cd cotpt
!pip install -q -e ".[all]"
!pip install -q -e ".[lora]"

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Test Zero-Shot Hidden Deliberation
For every visible token emitted, the model generates hidden thought tokens (shown in dimmed brackets `⟪...⟫`), uses them to pick the next real token, and immediately evicts them from the KV cache.

In [ ]:
from cotpt.model_utils import load_model_and_tokenizer, pick_device
from cotpt.inference import generate_with_hidden_deliberation
from cotpt.data import DEFAULT_PROMPT

device = pick_device()
model_id = "Qwen/Qwen3-0.6B"
print(f"Loading {model_id} on {device}...")
model, tokenizer = load_model_and_tokenizer(model_id, device)

prompt = DEFAULT_PROMPT
print("Running generation with per-token deliberation (5 thoughts/token):\n")
output = generate_with_hidden_deliberation(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_visible_tokens=60,
    num_hidden_tokens=5,
    show_hidden_thoughts=True,
)

## 3. Train on Reasoning Data (REINFORCE + LoRA + Value Baseline)
Fine-tune the model so its hidden thoughts learn to perform scratchpad calculation via REINFORCE on `gsm8k_direct`.

In [ ]:
!python scripts/train.py \
    --dataset-name gsm8k_direct \
    --use-lora \
    --num-steps 50 \
    --num-rollouts 3 \
    --num-think-positions 3 \
    --output-dir ./checkpoints/colab-cotpt

## 4. Run Benchmark: Normal vs COTPT
Compare:
- `normal_direct`: Direct response without scratchpad.
- `normal_cot`: Visible "Let's think step by step" (high peak KV cache).
- `cotpt_hidden`: Per-token hidden deliberation (bounded KV cache).
- `cotpt_adaptive`: Deliberation with trained MixingHead and entropy gating.

In [ ]:
!python scripts/benchmark.py \
    --model-id ./checkpoints/colab-cotpt \
    --dataset-name gsm8k \
    --limit 5 \
    --use-mixing-head \
    --conditions normal_direct,normal_cot,cotpt_hidden,cotpt_adaptive

## 5. Interactive Chat in Colab
Chat with your trained model right inside the notebook using the multi-turn formatting.

In [ ]:
from cotpt.model_utils import load_model_and_tokenizer, pick_device
from cotpt.inference import generate_with_hidden_deliberation
from main import format_chat_prompt

checkpoint_path = "./checkpoints/colab-cotpt"
device = pick_device()
model, tokenizer = load_model_and_tokenizer(checkpoint_path, device)

conversation = []
def chat(user_msg, num_hidden=5, show_thoughts=True):
    conversation.append({"role": "user", "content": user_msg})
    prompt = format_chat_prompt(tokenizer, conversation)
    print("Assistant: ", end="", flush=True)
    reply = generate_with_hidden_deliberation(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        max_visible_tokens=80,
        num_hidden_tokens=num_hidden,
        show_hidden_thoughts=show_thoughts,
        print_prompt=False,
    )
    conversation.append({"role": "assistant", "content": reply.strip()})
    return reply

chat("A jacket costs $80. It is on a 25% discount. What is the final price?")